# 15.7 Reinforcement Learning Basics

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 15: Responsible AI & Future Trends

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-15-responsible-ai-and-future-trends/15.07_reinforcement-learning-basics.ipynb)

Companion code for Section 15.7 of the book. Run the cells in order from top to bottom.

**Example 15.7.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# ── Q-Learning from scratch ───────────────────────────────────────
class QLearningAgent:
    '''
    Tabular Q-Learning agent for discrete environments.
    Learns Q(s,a) values by interacting with environment.
    '''

    def __init__(self, n_states, n_actions, alpha=0.1, gamma=0.99, epsilon=1.0,
                  epsilon_decay=0.995, epsilon_min=0.01):
        self.n_actions     = n_actions
        self.alpha         = alpha        # Learning rate
        self.gamma         = gamma        # Discount factor
        self.epsilon       = epsilon      # Exploration rate
        self.epsilon_decay = epsilon_decay
        self.epsilon_min   = epsilon_min
        self.Q             = defaultdict(lambda: np.zeros(n_actions))
        self.training_rewards = []

    def select_action(self, state) -> int:
        '''Epsilon-greedy action selection.'''
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)   # Explore
        return np.argmax(self.Q[state])                # Exploit

    def update(self, state, action, reward, next_state, done):
        '''Q-learning update: Q(s,a) ← Q(s,a) + α[r + γ max_a Q(s',a) - Q(s,a)]'''
        current_q = self.Q[state][action]
        if done:
            target_q = reward
        else:
            target_q = reward + self.gamma * np.max(self.Q[next_state])
        self.Q[state][action] += self.alpha * (target_q - current_q)

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)


# ── GridWorld environment ─────────────────────────────────────────
class GridWorld:
    '''Simple 5×5 grid: agent navigates from start to goal avoiding walls.'''

    def __init__(self, size=5):
        self.size  = size
        self.start = (0, 0)
        self.goal  = (size-1, size-1)
        self.walls = {(1,1),(1,2),(2,3),(3,1),(3,3)}
        self.state = self.start

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action):
        '''Actions: 0=up, 1=right, 2=down, 3=left'''
        r, c = self.state
        moves = [(-1,0),(0,1),(1,0),(0,-1)]
        dr, dc = moves[action]
        nr, nc = r+dr, c+dc

        # Stay in bounds and avoid walls
        if 0 <= nr < self.size and 0 <= nc < self.size and (nr,nc) not in self.walls:
            self.state = (nr, nc)

        done   = self.state == self.goal
        reward = 10 if done else (-5 if (self.state[0]+self.state[1]==0 and action>0) else -0.1)
        return self.state, reward, done


# ── Train Q-Learning agent ────────────────────────────────────────
env   = GridWorld(size=5)
agent = QLearningAgent(n_states=25, n_actions=4, alpha=0.1, gamma=0.95)

n_episodes = 2000
episode_rewards = []
episode_lengths = []

for episode in range(n_episodes):
    state  = env.reset()
    total_reward = 0
    steps = 0

    for step in range(100):  # Max 100 steps per episode
        action = agent.select_action(state)
        next_state, reward, done = env.step(action)
        agent.update(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
        steps += 1
        if done: break

    agent.decay_epsilon()
    episode_rewards.append(total_reward)
    episode_lengths.append(steps)

# Rolling average
window = 100
rolling_reward = [np.mean(episode_rewards[max(0,i-window):i+1]) for i in range(n_episodes)]

print('=== Q-Learning on GridWorld (5×5) ===')
print(f'Episodes: {n_episodes}')
print(f'Final epsilon: {agent.epsilon:.4f}')
print(f'Avg reward (last 100 episodes): {np.mean(episode_rewards[-100:]):.2f}')
print(f'Avg steps  (last 100 episodes): {np.mean(episode_lengths[-100:]):.1f}')
print(f'Q-table size: {len(agent.Q)} state-action pairs learned')

# Visualise learned Q-values
print('\nLearned Q-values at each state (best action):')
action_names = ['↑','→','↓','←']
for r in range(5):
    row_str = '  '
    for c in range(5):
        s = (r,c)
        if s == (2,2): row_str += 'G '  # Goal
        elif s in {(1,1),(1,2),(2,3),(3,1),(3,3)}: row_str += '■ '  # Wall
        else:
            best_action = np.argmax(agent.Q[s])
            row_str += action_names[best_action] + ' '
    print(row_str)

# ── RL Terms Reference ────────────────────────────────────────────
print('\n=== Key RL Algorithms ===')
rl_algorithms = {
    'Q-Learning':       'Off-policy tabular; learns Q* directly from experience',
    'SARSA':            'On-policy tabular; safer than Q-learning in stochastic envs',
    'DQN':              'Q-learning + neural network function approximation (Atari)',
    'DDPG':             'Continuous actions; actor-critic; off-policy',
    'PPO':              'Proximal Policy Optimisation; most widely used policy gradient',
    'SAC':              'Soft Actor-Critic; entropy-regularised; sample-efficient',
    'RLHF':             'Reinforcement Learning from Human Feedback; trains LLMs (ChatGPT)',
    'AlphaZero':        'MCTS + self-play; achieved superhuman chess/Go/shogi',
}
for name, desc in rl_algorithms.items():
    print(f'  {name:<15}: {desc}')